In [6]:
import os
import numpy as np
import urllib.request


class PneumoniaClusterBuilder:

    def __init__(self, out_dir="clusters", seed=42):

        self.out_dir = out_dir
        self.seed = seed
        os.makedirs(out_dir, exist_ok=True)

        self.client_id = os.getenv("CLIENT_ID", "B")
        self.total_clients = int(os.getenv("NUM_CLIENTS", "2"))
        self.client_ratios = os.getenv("CLIENT_RATIOS", None)

        self.val_ratio = float(os.getenv("VAL_RATIO", "0.2"))

        # dataset path
        self.dataset_path = os.getenv(
            "MEDMNIST_PATH",
            os.path.expanduser("~/.medmnist/pneumoniamnist.npz")
        )

        self.dataset_url = "https://zenodo.org/records/10519652/files/pneumoniamnist.npz?download=1"

    # ---------- download dataset ----------
    def _download_dataset(self):

        if os.path.exists(self.dataset_path):
            return

        os.makedirs(os.path.dirname(self.dataset_path), exist_ok=True)

        print("Downloading PneumoniaMNIST dataset...")
        urllib.request.urlretrieve(self.dataset_url, self.dataset_path)

        print("Dataset downloaded:", self.dataset_path)

    # ---------- load dataset ----------
    def _load_dataset(self):

        self._download_dataset()

        data = np.load(self.dataset_path)

        Xtr = data["train_images"]
        ytr = data["train_labels"]

        Xv = data["val_images"]
        yv = data["val_labels"]

        # convert shape -> (N,1,28,28)
        Xtr = Xtr[:, None, :, :].astype(np.float32) / 255
        Xv = Xv[:, None, :, :].astype(np.float32) / 255

        return Xtr, ytr, Xv, yv

    # ---------- stratified split ----------
    def _stratified_split(self, X_all, y_all):

        rng = np.random.default_rng(self.seed)

        if self.client_ratios:
            ratios = np.array([float(x) for x in self.client_ratios.split(",")])
            ratios = ratios / ratios.sum()
        else:
            ratios = np.ones(self.total_clients) / self.total_clients

        splits = [[] for _ in range(self.total_clients)]

        labels = np.unique(y_all)

        for label in labels:

            label_idx = np.where(y_all == label)[0]
            rng.shuffle(label_idx)

            n_label = len(label_idx)

            sizes = (ratios * n_label).astype(int)
            sizes[-1] = n_label - sizes[:-1].sum()

            start = 0
            for cid in range(self.total_clients):
                end = start + sizes[cid]
                splits[cid].extend(label_idx[start:end])
                start = end

        splits = [np.array(s) for s in splits]

        return splits

    # ---------- main ----------
    def build_for_env(self):

        out_path = f"{self.out_dir}/cluster_client_{self.client_id}.npz"

        if os.path.exists(out_path):
            print(f"[CLIENT {self.client_id}] Exists → skip")
            return

        print(f"[CLIENT {self.client_id}] Building dataset")

        Xtr, ytr, Xv, yv = self._load_dataset()

        X_all = np.concatenate([Xtr, Xv])
        y_all = np.concatenate([ytr, yv])

        splits = self._stratified_split(X_all, y_all)

        cid_list = [chr(ord("A") + i) for i in range(self.total_clients)]
        cid_to_idx = dict(zip(cid_list, splits))

        my_idx = cid_to_idx[self.client_id]

        X_client = X_all[my_idx]
        y_client = y_all[my_idx]

        # ---------- split train / eval ----------
        rng = np.random.default_rng(self.seed)

        perm = rng.permutation(len(X_client))

        n_val = int(len(X_client) * self.val_ratio)

        val_idx = perm[:n_val]
        train_idx = perm[n_val:]

        X_train = X_client[train_idx]
        y_train = y_client[train_idx]

        X_eval = X_client[val_idx]
        y_eval = y_client[val_idx]

        np.savez_compressed(
            out_path,
            X_train=X_train,
            y_train=y_train,
            X_eval=X_eval,
            y_eval=y_eval
        )

        print(f"[CLIENT {self.client_id}]")
        print("Train:", len(X_train))
        print("Eval :", len(X_eval))

    # ---------- loader ----------
    @staticmethod
    def load_my_cluster(out_dir="clusters"):

        cid = os.getenv("CLIENT_ID", "A")
        path = f"{out_dir}/cluster_client_{cid}.npz"

        data = np.load(path)

        return (
            data["X_train"],
            data["y_train"],
            data["X_eval"],
            data["y_eval"],
        )


if __name__ == "__main__":

    role = os.getenv("ROLE", "client").lower()
    if role != "client":
        raise RuntimeError("CLIENT only")

    builder = PneumoniaClusterBuilder(
        out_dir=os.getenv("OUT_DIR", "../clusters"),
        seed=int(os.getenv("DATA_SEED", "2")),
    )

    builder.build_for_env()

[CLIENT B] Building dataset
[CLIENT B]
Train: 2094
Eval : 523


In [8]:
#check data
import numpy as np
import matplotlib.pyplot as plt

# đường dẫn file
file_path = r"D:\DHCN\2025-2026\HK2\CongNgheMoi\project\clients\clusters\cluster_client_A.npz"

# load file
data = np.load(file_path)

# xem các key trong file
print("Keys in file:", data.files)

# kiểm tra từng key
for key in data.files:
    arr = data[key]
    print(f"\nKey: {key}")
    print("Shape:", arr.shape)
    print("Type:", arr.dtype)

# giả sử dataset có images và labels
images = data['images'] if 'images' in data.files else None
labels = data['labels'] if 'labels' in data.files else None

# kiểm tra label distribution
if labels is not None:
    unique, counts = np.unique(labels, return_counts=True)
    print("\nLabel distribution:")
    for u, c in zip(unique, counts):
        print(f"Label {u}: {c} samples")

# hiển thị vài ảnh
if images is not None:
    plt.figure(figsize=(8,4))
    for i in range(8):
        plt.subplot(2,4,i+1)
        plt.imshow(images[i].squeeze(), cmap="gray")
        if labels is not None:
            plt.title(f"Label: {labels[i]}")
        plt.axis("off")
    plt.show()

Keys in file: ['X_train', 'y_train', 'X_eval', 'y_eval']

Key: X_train
Shape: (2092, 1, 28, 28)
Type: float32

Key: y_train
Shape: (2092, 1)
Type: uint8

Key: X_eval
Shape: (523, 1, 28, 28)
Type: float32

Key: y_eval
Shape: (523, 1)
Type: uint8


In [25]:
import json
import os
import numpy as np

CLUSTER_DIR = "../clusters"
STREAM_DIR = "data_stream"


# ================= utils =================

def _get_client_id():
    cid = os.getenv("CLIENT_ID","A")
    if not cid:
        raise ValueError("CLIENT_ID env missing")
    return cid


def _cluster_path(cid):
    return os.path.join(CLUSTER_DIR, f"cluster_client_{cid}.npz")


def _stream_path(cid):
    os.makedirs(STREAM_DIR, exist_ok=True)
    return os.path.join(STREAM_DIR, f"stream_client_{cid}.npz")


def _pointer_path(cid):
    return os.path.join(STREAM_DIR, f"stream_pointer_{cid}.txt")


# ================= load pointer =================

def _load_pointer(path):
    if not os.path.exists(path):
        return {"train": 0, "eval": 0}

    with open(path, "r") as f:
        return json.load(f)


def _save_pointer(path, ptr):
    with open(path, "w") as f:
        json.dump(ptr, f)


# ================= main generator =================

def generate_batch(batch_size=50):

    cid = _get_client_id()

    cluster_file = _cluster_path(cid)
    stream_file = _stream_path(cid)
    pointer_file = _pointer_path(cid)

    if not os.path.exists(cluster_file):
        raise FileNotFoundError(f"Cluster file missing: {cluster_file}")

    data = np.load(cluster_file)

    X_train = data["X_train"]
    y_train = data["y_train"]

    X_eval = data["X_eval"]
    y_eval = data["y_eval"]

    ptr = _load_pointer(pointer_file)

    train_ptr = int(ptr["train"])
    eval_ptr = int(ptr["eval"])

    total_train = len(X_train)
    total_eval = len(X_eval)

    # ===== train batch =====
    if train_ptr >= total_train:
        X_new_train = np.empty((0, *X_train.shape[1:]))
        y_new_train = np.empty((0, *y_train.shape[1:]))
    else:
        end = min(train_ptr + batch_size, total_train)
        X_new_train = X_train[train_ptr:end]
        y_new_train = y_train[train_ptr:end]
        train_ptr = end

    # ===== eval batch =====
    if eval_ptr >= total_eval:
        X_new_eval = np.empty((0, *X_eval.shape[1:]))
        y_new_eval = np.empty((0, *y_eval.shape[1:]))
    else:
        end = min(eval_ptr + batch_size, total_eval)
        X_new_eval = X_eval[eval_ptr:end]
        y_new_eval = y_eval[eval_ptr:end]
        eval_ptr = end

    # nếu cả train và eval đều rỗng → dataset exhausted
    if len(X_new_train) == 0 and len(X_new_eval) == 0:
        print(f"[STREAM] client={cid} dataset exhausted → no new data")
        return

    ptr = {
        "train": train_ptr,
        "eval": eval_ptr
        }

    _save_pointer(pointer_file, ptr)

    # ===== append stream =====
    if os.path.exists(stream_file):

        old = np.load(stream_file)

        Xs_train = np.concatenate([old["X_train"], X_new_train])
        ys_train = np.concatenate([old["y_train"], y_new_train])

        Xs_eval = np.concatenate([old["X_eval"], X_new_eval])
        ys_eval = np.concatenate([old["y_eval"], y_new_eval])

    else:

        Xs_train = X_new_train
        ys_train = y_new_train

        Xs_eval = X_new_eval
        ys_eval = y_new_eval

    np.savez(
        stream_file,
        X_train=Xs_train,
        y_train=ys_train,
        X_eval=Xs_eval,
        y_eval=ys_eval
    )

    print(
        f"[STREAM] client={cid} "
        f"train+={len(X_new_train)} "
        f"eval+={len(X_new_eval)} "
        f"train_ptr={train_ptr}/{total_train} "
        f"eval_ptr={eval_ptr}/{total_eval}"
    )

# ================= run =================

if __name__ == "__main__":

    batch_size = int(os.getenv("STREAM_BATCH", "500"))

    generate_batch(batch_size)


[STREAM] client=A dataset exhausted → no new data


In [ ]:
#check_duplicates_data
import numpy as np

file_path = r"D:\DHCN\2025-2026\HK2\CongNgheMoi\project\clients\data\data_stream\stream_client_A.npz"

data = np.load(file_path)

X_train = data["X_train"]
y_train = data["y_train"]

X_eval = data["X_eval"]
y_eval = data["y_eval"]


def check_duplicates(X, name):

    # flatten để so sánh
    X_flat = X.reshape(X.shape[0], -1)

    unique_rows = np.unique(X_flat, axis=0)

    total = len(X)
    unique = len(unique_rows)

    print(f"\n{name}")
    print("total samples:", total)
    print("unique samples:", unique)
    print("duplicates:", total - unique)


check_duplicates(X_train, "TRAIN DATA")
check_duplicates(X_eval, "EVAL DATA")


TRAIN DATA
total samples: 2092
unique samples: 2090
duplicates: 2

EVAL DATA
total samples: 523
unique samples: 522
duplicates: 1


In [7]:
#check data
import numpy as np
import matplotlib.pyplot as plt

# đường dẫn file
file_path = r"D:\DHCN\2025-2026\HK2\CongNgheMoi\project\clients\data\data_stream\stream_client_A.npz"

# load file
data = np.load(file_path)

# xem các key trong file
print("Keys in file:", data.files)

# kiểm tra từng key
for key in data.files:
    arr = data[key]
    print(f"\nKey: {key}")
    print("Shape:", arr.shape)
    print("Type:", arr.dtype)

# giả sử dataset có images và labels
images = data['images'] if 'images' in data.files else None
labels = data['labels'] if 'labels' in data.files else None

# kiểm tra label distribution
if labels is not None:
    unique, counts = np.unique(labels, return_counts=True)
    print("\nLabel distribution:")
    for u, c in zip(unique, counts):
        print(f"Label {u}: {c} samples")

# hiển thị vài ảnh
if images is not None:
    plt.figure(figsize=(8,4))
    for i in range(8):
        plt.subplot(2,4,i+1)
        plt.imshow(images[i].squeeze(), cmap="gray")
        if labels is not None:
            plt.title(f"Label: {labels[i]}")
        plt.axis("off")
    plt.show()

Keys in file: ['X_train', 'y_train', 'X_eval', 'y_eval']

Key: X_train
Shape: (2092, 1, 28, 28)
Type: float32

Key: y_train
Shape: (2092, 1)
Type: uint8

Key: X_eval
Shape: (523, 1, 28, 28)
Type: float64

Key: y_eval
Shape: (523, 1)
Type: float64


In [ ]:
# load stream data
import os
import numpy as np
import torch
from torch.utils.data import TensorDataset


STREAM_DIR = "data_stream"


# ================= path resolve (CLIENT ONLY) =================

def _get_client_stream_path():

    role = os.getenv("ROLE", "client").lower()
    if role != "client":
        raise RuntimeError("This loader is CLIENT-only")

    client_id = os.getenv("CLIENT_ID",'A')
    if not client_id:
        raise ValueError("CLIENT_ID env is required")

    name = f"stream_client_{client_id}.npz"
    path = os.path.join(STREAM_DIR, name)

    if not os.path.exists(path):
        raise FileNotFoundError(
            f"[DATA] Stream dataset not found: {path}\n"
            f"→ Wait for Airflow ingest_new_data to generate stream first"
        )

    return path


# ================= tensor convert =================

def _to_tensor(X, y):
    X = torch.tensor(X, dtype=torch.float32)
    y = torch.tensor(y, dtype=torch.float32)

    if y.ndim == 1:
        y = y.unsqueeze(1)

    return X, y


def load_data_split(client_id,seed):


    seed_env = os.getenv("DATA_SEED",'1')
    if seed_env:
        seed = int(seed_env)
    path = _get_client_stream_path()
    data = np.load(path)

    if "X_train" not in data:
        raise ValueError(f"{path} is not a valid stream dataset")

    X_train, y_train = _to_tensor(data["X_train"], data["y_train"])
    X_eval, y_eval = _to_tensor(data["X_eval"], data["y_eval"])



    train_ds = TensorDataset(X_train, y_train)
    val_ds   = TensorDataset(X_eval, y_eval)

    print(
        f"[DATA][CLIENT] id={os.getenv('CLIENT_ID','A')} "
        f"stream={os.path.basename(path)} "
        f"train={len(train_ds)} val={len(val_ds)}"
    )

    return train_ds, val_ds



[DATA][CLIENT] id=A stream=stream_client_A.npz train=2092 val=523


(<torch.utils.data.dataset.TensorDataset at 0x250de915340>,
 <torch.utils.data.dataset.TensorDataset at 0x250de917500>)